# Track A, Session 2, Models, Evaluation and XAI (your turn)
Fill the `# TODO` lines and the **Observe** cells. Question of the session: does a learned model beat the baselines, and does it rely on the signals we read in Part A?

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error
plt.rcParams.update({"figure.figsize": (11, 3.6), "axes.titleweight": "bold"})
BLUE, ORANGE, GREEN, GREY = "#1f6feb", "#fb8500", "#2a9d8f", "#9aa0a6"
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC = ROOT / "data" / "processed"; IMG = ROOT / "images" / "partC"; IMG.mkdir(parents=True, exist_ok=True)
MODELS = ROOT / "models"; MODELS.mkdir(exist_ok=True)

TARGET, REFERENCE = "total load actual", "total load forecast"
data = pd.read_csv(PROC / "trackA_features.csv", parse_dates=["time"])
FEATURES = [c for c in data.columns if c not in ("time", TARGET, REFERENCE)]
year = data["time"].dt.year
train, val, test = data[year <= 2016], data[year == 2017], data[year == 2018]
X_train, y_train = train[FEATURES], train[TARGET]
X_val,   y_val   = val[FEATURES],   val[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

def evaluate(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "MAPE_%": float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)}

print(len(FEATURES), "features | train/val/test:", len(train), len(val), len(test))

## 0. The models we will use, in plain words
A model learns a function from the features to the demand: it sees many examples (features to demand) and finds the pattern that links them.

- **Ridge, a linear model.** It predicts demand as a weighted sum of the features: demand is roughly w1 * feature1 + w2 * feature2 + ... Each weight says how strongly a feature pushes demand up or down. Ridge adds a small penalty on large weights so it does not overfit. It is transparent (you can read the weights), but it needs the features on comparable scales, which is why it sits in a pipeline with a scaler.
- **Decision tree.** It asks a cascade of yes/no questions on the features (is hour above 18? is temperature above 25?) and predicts the average demand in each final group. It captures non-linear effects and interactions on its own, and needs no scaling.
- **Gradient boosting, HistGradientBoosting and XGBoost.** Hundreds of small trees in sequence, each one correcting the errors of the previous ones. It is the strong default for tabular data, and the reason the tree models beat the linear one here.

We train one linear model and one boosting model, then compare them to the baselines.

## 1. Ridge, a transparent linear model
Ridge needs scaling, so it lives in a pipeline (imputer, scaler, Ridge) fitted on train only. **Your turn:** fit it and evaluate on validation.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

ridge = Pipeline([("imputer", SimpleImputer(strategy="median")),
                  ("scaler", StandardScaler()),
                  ("model", Ridge(alpha=1.0))])
...   # TODO: fit on train
print("Ridge val:", {k: round(v, 2) for k, v in evaluate(y_val, ridge.predict(X_val)).items()})

## Look inside Ridge: it is a weighted sum
Ridge learned one weight per feature. On the scaled features, a large positive weight means the feature pushes demand up, a large negative weight pulls it down. This is what "linear and transparent" means: we can read the model directly.

In [ ]:
coefs = pd.Series(ridge.named_steps["model"].coef_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(9, 5))
coefs.plot.barh(ax=ax, color=[ORANGE if c < 0 else BLUE for c in coefs])
ax.set_title("Ridge weight per feature (scaled): blue pushes demand up, orange pulls it down")
ax.set_xlabel("weight")
fig.savefig(IMG / "ridge_coefficients.png", dpi=130, bbox_inches="tight"); plt.show()

## 2. HistGradientBoosting, a non-linear tabular model
Trees capture non-linear effects (the temperature curve). **Your turn:** fit it and evaluate on validation.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
hgb = HistGradientBoostingRegressor(random_state=42)
...   # TODO: fit on train
print("HistGB val:", {k: round(v, 2) for k, v in evaluate(y_val, hgb.predict(X_val)).items()})

## Look inside a tree: the questions it asks
A boosting model stacks hundreds of small trees. Here is ONE shallow tree so you can see how a tree works: each box is a yes/no question on a feature (for example, is load_lag_24 below some value?), the data goes left or right, and each leaf predicts the average demand of the examples that reached it. The full model asks such questions hundreds of times and adds up the answers.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree
demo_tree = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_train, y_train)
fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(demo_tree, feature_names=FEATURES, filled=True, rounded=True,
          impurity=False, precision=0, fontsize=8, ax=ax)
ax.set_title("One small decision tree (depth 3): the yes/no questions it asks, and the demand predicted in each leaf")
fig.savefig(IMG / "decision_tree.png", dpi=130, bbox_inches="tight"); plt.show()

## 3. Optional extension: XGBoost
Same idea as HistGB, a strong gradient boosting library. Its settings play the same roles: n_estimators, the number of trees; learning_rate, the step size; max_depth, the tree depth; and subsample / colsample_bytree, which add a bit of randomness to avoid overfitting. We chose reasonable values but did not tune them; tuning is the next step. Optional: the cell skips cleanly if the package is not installed.

In [ ]:
try:
    from xgboost import XGBRegressor
    xgb = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                       subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=-1)
    xgb.fit(X_train, y_train)
    print("XGBoost val:", {k: round(v, 2) for k, v in evaluate(y_val, xgb.predict(X_val)).items()})
except ImportError:
    xgb = None
    print("xgboost not installed: skipping the optional extension")

## 4. Final comparison on the untouched test year
The 2018 test set is used once, here. Baselines, reference, and models in one honest table.

In [ ]:
ref = test.dropna(subset=[REFERENCE])
rows = [
    {"model": "Naive: same hour yesterday",   **evaluate(y_test, test["load_lag_24"])},
    {"model": "Naive: same hour last week",   **evaluate(y_test, test["load_lag_168"])},
    {"model": "Provided forecast (reference)", **evaluate(ref[TARGET], ref[REFERENCE])},
    {"model": "Ridge",                         **evaluate(y_test, ridge.predict(X_test))},
    {"model": "HistGradientBoosting",          **evaluate(y_test, hgb.predict(X_test))},
]
if xgb is not None:
    rows.append({"model": "XGBoost", **evaluate(y_test, xgb.predict(X_test))})
results = pd.DataFrame(rows).set_index("model").round(2)
results

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
results["MAE"].sort_values(ascending=True).plot.barh(ax=ax, color=[GREEN]+[BLUE]*(len(results)-1))
ax.set_title("Test 2018: MAE by model (lower is better)"); ax.set_xlabel("MAE (MW)")
fig.savefig(IMG / "results_mae.png", dpi=130, bbox_inches="tight"); plt.show()

**Observe.** Do the models beat the naive baselines? Where does the best model stand relative to the provided forecast? Is that a failure?

## 5. Look at the forecasts, not only the scores
One test week, actual vs predicted, then the residual distribution.

In [ ]:
best_name, best_model = min(
    [("Ridge", ridge), ("HistGradientBoosting", hgb)] + ([("XGBoost", xgb)] if xgb is not None else []),
    key=lambda kv: evaluate(y_val, kv[1].predict(X_val))["MAE"])
print("best on validation:", best_name)

wk = test[(test["time"] >= "2018-03-05") & (test["time"] < "2018-03-12")]
fig, ax = plt.subplots()
ax.plot(wk["time"], wk[TARGET], color=BLUE, label="actual")
ax.plot(wk["time"], best_model.predict(wk[FEATURES]), color=ORANGE, ls="--", label=f"{best_name} (day-ahead)")
ax.set_title("Actual vs predicted, one test week"); ax.set_ylabel("MW"); ax.legend()
fig.savefig(IMG / "pred_week.png", dpi=130, bbox_inches="tight"); plt.show()

resid = y_test - best_model.predict(X_test)
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.hist(resid, bins=60, color=GREY)
ax.set_title(f"{best_name}: residuals on test (mean {resid.mean():.0f} MW)"); ax.set_xlabel("error (MW)")
fig.savefig(IMG / "residuals.png", dpi=130, bbox_inches="tight"); plt.show()

## 6. XAI core: permutation importance
Shuffle one feature at a time; if the score drops, the model relied on it. **Your turn:** run it on the validation year and plot the top features.

In [ ]:
from sklearn.inspection import permutation_importance
perm = ...   # TODO: permutation_importance(best_model, X_val, y_val, n_repeats=5, random_state=42, scoring="neg_mean_absolute_error")
imp = pd.Series(perm.importances_mean, index=FEATURES).sort_values()
imp.tail(12).plot.barh(color="#1f6feb")

## 7. Explainability, deeper: SHAP (optional extension)
Permutation importance told us WHICH features matter overall. SHAP goes further: for every single prediction it splits the result into the contribution of each feature. It answers a very concrete question, why did the model predict THIS value for THIS hour?

The idea: start from the average prediction over the data; each feature then pushes the prediction up or down; SHAP measures those pushes, and they add up exactly to the final prediction. We look at three views: one prediction explained, the whole model at once, and one feature in detail. The cell skips cleanly if SHAP is not installed.

In [ ]:
try:
    import shap
    explain_model = xgb if xgb is not None else hgb
    sample = X_val.sample(2000, random_state=42)
    sv = shap.TreeExplainer(explain_model)(sample)
    print("SHAP ready on", explain_model.__class__.__name__, "| explaining", sample.shape[0], "hours")
except Exception as e:
    sv = None; print("shap not available, skipping the optional extension:", type(e).__name__)

### 7a. One prediction explained (waterfall)
Read it from the bottom: E[f(x)] is the average prediction over the data. Each bar is one feature pushing THIS specific hour up (to the right) or down (to the left). The top, f(x), is the final prediction for that hour. This is SHAP's most useful view: it explains a single decision.

In [ ]:
if sv is not None:
    shap.plots.waterfall(sv[0], max_display=10, show=False)
    fig = plt.gcf(); fig.set_size_inches(9, 5)
    fig.savefig(IMG / "shap_waterfall.png", dpi=130, bbox_inches="tight"); plt.show()

### 7b. The whole model at once (beeswarm)
How to read it: each row is a feature, ordered by importance from top to bottom. Each dot is one prediction. Left or right means the feature pushed that prediction down or up. The colour is the feature value, red for high and blue for low. So for load_lag_24, red dots on the right read as: a high demand yesterday pushes today's forecast up.

In [ ]:
if sv is not None:
    shap.plots.beeswarm(sv, max_display=12, show=False)
    plt.gcf().savefig(IMG / "shap_beeswarm.png", dpi=130, bbox_inches="tight"); plt.show()

### 7c. One feature in detail (temperature)
Temperature on the x-axis, its SHAP contribution on the y-axis. This should reproduce the non-linear curve we read in Part A: little effect in mild weather, then a sharp rise above 25 C as cooling demand kicks in.

In [ ]:
if sv is not None:
    shap.plots.scatter(sv[:, "temp"], color=sv, show=False)
    plt.gcf().savefig(IMG / "shap_temp.png", dpi=130, bbox_inches="tight"); plt.show()

**Observe.** Do the important features match the Part A reading (lags, hour, temperature)? Does the temperature effect look like the curve from Part A? Any feature that should worry us about leakage?

## 8. Export for the backend
The FastAPI backend must reload the model, never retrain it. We save the model, the feature list and metadata together.

In [ ]:
artifact = {"model": best_model, "model_name": best_name, "features": FEATURES,
            "target": TARGET, "horizon": "t+24 (day-ahead)",
            "test_metrics": results.loc[best_name].to_dict(),
            "weather_assumption": "observed weather at target hour used as a day-ahead forecast proxy"}
joblib.dump(artifact, MODELS / "forecasting_model.joblib")
print("saved:", MODELS / "forecasting_model.joblib")

## Wrap-up
Part A read the signal. Part B protected the data. Today the reading became features, the features fed models, the models beat the naive baselines, and XAI confirmed the story. Next for the project: the full t+1 to t+24 output, the FastAPI backend that reloads these artifacts, and the Streamlit dashboard.